In [ ]:
'''
Overview
this notebook parses all past ufc fight stats when run, it does not include upcoming fights


scrape ufc fight stats
get all event details, name, url, date, location for all ufc events
for each event, get fight details all fights on card
parse each fight to get fight stats of both fighters
'''

import sys
sys.executable

In [ ]:


# imports
import pandas as pd
from tqdm.notebook import tqdm_notebook

# import library
import scrape_ufc_stats_library as LIB

# import config
import yaml
config = yaml.safe_load(open('scrape_ufc_stats_config.yaml'))

# Parse Event Details
Includes:
<br>
Event
<br>
URL
<br>
Date
<br>
Location
<br>

In [ ]:
# define url to parse
events_url = config['completed_events_all_url']

In [ ]:
# get soup
soup = LIB.get_soup(events_url)

# parse event details
all_event_details_df = LIB.parse_event_details(soup)

# show event details
display(all_event_details_df)

# write event details to file
all_event_details_df.to_csv(config['event_details_file_name'], index=False)

# Parse Fight Details
Includes:
<br>
Event
<br>
Bout
<br>
URL

In [ ]:
# define list of urls of fights to parse
list_of_events_urls = reversed(list(all_event_details_df['URL']))

In [ ]:
load_from_disk = True

if load_from_disk:
    all_fight_details_df = pd.read_csv(config['fight_details_file_name'])
    # filter to only events not yet in the loaded data
    parsed_event_urls = set(all_fight_details_df['EVENT_URL'])
    remaining_event_urls = [url for url in list_of_events_urls if url not in parsed_event_urls]
else:
    # create empty df to store fight details
    all_fight_details_df = pd.DataFrame(columns=config['fight_details_column_names'])
    remaining_event_urls = list_of_events_urls

# loop through each event and parse fight details
for (i, url) in enumerate(tqdm_notebook(remaining_event_urls)):

    # get soup
    soup = LIB.get_soup(url)

    # parse fight links
    fight_details_df = LIB.parse_fight_details(soup, url)
    
    # concat fight details
    all_fight_details_df = pd.concat([fight_details_df, all_fight_details_df])

    if i % 10 == 0:
        all_fight_details_df.to_csv(config['fight_details_file_name'], index=False)

# show all fight details
display(all_fight_details_df)

# write fight details to file
all_fight_details_df.to_csv(config['fight_details_file_name'], index=False)

# Parse Fight Results and Fight Stats

Fight Results Includes:
<br>
Event
<br>
Bout
<br>
Weightclass
<br>
Method
<br>
Round
<br>
Time
<br>
Time Format
<br>
Referee
<br>
Details
<br>

Fight Stats Includes:
<br>
Event
<br>
Bout
<br>
Round
<br>
Fighter
<br>
Kd
<br>
Sig.Str.
<br>
Sig.Str. %
<br>
Total Str.
<br>
Td
<br>
Td %
<br>
Sub.Att
<br>
Rev.
<br>
Ctrl
<br>
Head
<br>
Body
<br>
Leg
<br>
Distance
<br>
Clinch
<br>
Ground
<br>

In [ ]:
# define list of urls of fights to parse
list_of_fight_details_urls = reversed(list(all_fight_details_df['URL']))

In [ ]:
load_results_from_disk = True

if load_results_from_disk:
    all_fight_results_df = pd.read_csv(config['fight_results_file_name'])
    all_fight_stats_df = pd.read_csv(config['fight_stats_file_name'])
    parsed_fight_urls = set(all_fight_results_df['URL'])
    remaining_fight_urls = [url for url in list_of_fight_details_urls if url not in parsed_fight_urls]
else:
    # create empty df to store fight results
    all_fight_results_df = pd.DataFrame(columns=config['fight_results_column_names'])
    # create empty df to store fight stats
    all_fight_stats_df = pd.DataFrame(columns=config['fight_stats_column_names'])
    remaining_fight_urls = list_of_fight_details_urls

# build fight_url -> event_url lookup
fight_to_event_url = dict(all_fight_details_df[['URL', 'EVENT_URL']].values)

# loop through each fight and parse fight results and stats
for (i, url) in enumerate(tqdm_notebook(remaining_fight_urls)):

    # get soup
    soup = LIB.get_soup(url)
    event_url = fight_to_event_url.get(url, '')

    # parse fight results and fight stats
    fight_results_df, fight_stats_df = LIB.parse_organise_fight_results_and_stats(
        soup,
        url,
        event_url,
        config['fight_results_column_names'],
        config['totals_column_names'],
        config['significant_strikes_column_names']
        )

    # concat fight results
    all_fight_results_df = pd.concat([fight_results_df, all_fight_results_df])
    # concat fight stats
    all_fight_stats_df = pd.concat([fight_stats_df, all_fight_stats_df])

    if i % 50 == 0:
        all_fight_results_df.to_csv(config['fight_results_file_name'], index=False)
        all_fight_stats_df.to_csv(config['fight_stats_file_name'], index=False)

# show all fight results
display(all_fight_results_df)
# show all fight stats
display(all_fight_stats_df)

# write to file
all_fight_results_df.to_csv(config['fight_results_file_name'], index=False)
# write to file
all_fight_stats_df.to_csv(config['fight_stats_file_name'], index=False)